# Michelson interferometer phase: profiling a fringe-frequency nuisance

This notebook accompanies the docs page [`michelson-phase`](../../docs/examples/michelson-phase.md) and the portal walkthrough. A Michelson interferometer counts photons along the fringe coordinate $u = kx$; the phase $\varphi$ is the parameter of interest and a fractional fringe-frequency error $\epsilon$ is the nuisance a short baseline cannot avoid confounding with it. The conditional score is closed-form at the reference point $(\varphi_0,\epsilon_0)=(0,0)$, so this notebook is a check on the library as much as a demonstration of it. It runs on `ExecutionConfig(backend="numpy", precision="float64", device="cpu")` throughout -- the analytic `ScoreFunction` route against a bounded `IntegrationSource`.

The story has two acts: ordinary D-optimal quantization of the two-dimensional score, then the profiled criterion for the phase alone, with the fragmentation of the profiled partition diagnosed rather than hidden.

## Data

`examples.michelson_phase` builds the score provider and a deterministic midpoint-quadrature sample. Node count shrinks under `SCOREQUANT_EXAMPLE_FAST` through `example_scale`, so this notebook runs quickly in CI and at full research scale locally.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples._env import example_scale
from examples.michelson_phase import (
    EXECUTION,
    INTEREST,
    build_integration_source,
    build_provider,
    build_train_sample,
    closed_form_information,
    equal_width_labels,
    make_d_geometry_figure,
    make_profiled_figure,
    profiled_retention,
    profiled_trace,
    reusable_rules,
    run_diagnostics,
    run_study,
    smoothing_ladder,
    sweep_bin_budget,
    unbinned_profiled_information,
)

n_nodes = example_scale(8_000, 1_000)
soft_steps = example_scale(1_000, 40)
budgets = example_scale((4, 6, 8, 10), (4, 6))
headline_bins = 6

provider = build_provider()
sample = build_train_sample(provider, n_nodes=n_nodes)
sample.scores.shape, sample.weights.sum()

## Closed forms

Over whole fringes the normalizer's $\varphi$-derivative vanishes at the reference point but its $\epsilon$-derivative does not, which is exactly what fixes the $-V$ term in $s_\epsilon(u) = u\,s_\varphi(u) - V$ -- not a centering convenience, but the reason $E[s_\epsilon]=0$ holds exactly. `fringe_density` and `michelson_score` are periodic in $u$ up to that one explicit term, so midpoint quadrature of the periodic part converges exponentially: `fisher_information` reproduces $I_{\varphi\varphi}=1-\sqrt{1-V^2}$ and $I_{\varphi\epsilon}=I_{\varphi\varphi}\,u_{\max}/2$ to machine precision.

In [ ]:
information = np.asarray(sq.fisher_information(sample.scores, sample.weights, execution=EXECUTION))
closed_form = closed_form_information()

assert abs(float(information[0, 0]) - closed_form["i_phiphi"]) < 1e-12
assert abs(float(information[0, 1]) - closed_form["i_phieps"]) < 1e-12

i_phiphi = float(information[0, 0])
i_phieps = float(information[0, 1])
i_epseps = float(information[1, 1])
correlation = i_phieps / np.sqrt(i_phiphi * i_epseps)
i_phiphi, i_phieps, i_epseps, correlation

## The profiled ceiling

Every phase-retention number below is stated against the *unbinned profiled* information -- the Schur complement of the nuisance column out of `fisher_information` -- never against $I_{\varphi\varphi}$, which is not available to an analyst who does not know $\epsilon$.

In [ ]:
reference = unbinned_profiled_information(sample.scores, sample.weights)
cost_of_profiling = 1.0 - reference / i_phiphi
reference, cost_of_profiling

## Three partitions, and the aliasing hazard

`optimize_partition` runs on `provider.score(X)` for both finite criteria; the naive equal-width detector segmentation is scored the same way for a fair comparison. Four equal segments over four whole fringes make each segment exactly one period, so the naive rule's between-cell matrix is rank-deficient and it retains *exactly* nothing of the phase at $K=4$.

In [ ]:
rows = []
for n_bins in budgets:
    equal_labels = equal_width_labels(sample.observations, n_bins)
    d_partition = sq.optimize_partition(
        sample.scores,
        weights=sample.weights,
        n_bins=n_bins,
        criterion=sq.DOptimality(),
        config=sq.DExchangeConfig(seed=4),
        execution=EXECUTION,
    )
    bound = sq.efficient_score_bound(
        sample.scores, interest=INTEREST, weights=sample.weights, n_bins=n_bins, execution=EXECUTION
    )
    profiled_partition = sq.optimize_partition(
        sample.scores,
        weights=sample.weights,
        n_bins=n_bins,
        criterion=sq.ProfiledDOptimality(interest=INTEREST),
        config=sq.DExchangeConfig(seed=4),
        initial_labels=bound.labels,
        execution=EXECUTION,
    )
    rows.append(
        (
            n_bins,
            profiled_retention(sample, equal_labels, n_bins),
            profiled_retention(sample, np.asarray(d_partition.labels), n_bins),
            profiled_retention(sample, np.asarray(profiled_partition.labels), n_bins),
            bound.gap_to(profiled_partition),
        )
    )
    if n_bins == headline_bins:
        headline_d_partition, headline_profiled_partition = d_partition, profiled_partition

print(f"{'K':>3}  {'equal-width':>11}  {'D-optimal':>9}  {'profiled Ds':>11}  {'bound gap':>10}")
for n_bins, equal, d_opt, ds_opt, gap in rows:
    print(f"{n_bins:>3}  {equal:>11.4f}  {d_opt:>9.4f}  {ds_opt:>11.4f}  {gap:>10.2e}")

assert rows[0][1] < 1e-6  # K=4 equal-width segments retain nothing

## The compile bridge, and the profiled refusal

The `DOptimality` partition is exchange-stable, so `compile_quantizer()` turns it into a Mahalanobis rule that reproduces its own training labels. The profiled partition has no canonical inductive rule to compile into, and `compile_quantizer()` refuses with a named counterexample instead of guessing.

In [ ]:
assert headline_d_partition.exchange_stable is True
compiled = headline_d_partition.compile_quantizer(execution=EXECUTION)

try:
    headline_profiled_partition.compile_quantizer(execution=EXECUTION)
    refusal_message = None
except sq.RefusalError as error:
    refusal_message = str(error)

assert refusal_message is not None and refusal_message.endswith("[CE-DS-GLOBAL-GEOMETRY-001]")
refusal_message

## The reusable rule on the missing route

`fit_quantizer(source, provider=provider, ...)` with `source` the bounded `IntegrationSource` built at the top of this notebook -- the analytic-score-against-quadrature-measure route. A compiled `DOptimality` rule has `hardening_gap == 0.0` by construction; the soft `ProfiledDOptimality` fit is the only route to a *reusable* profiled rule at all, since finite profiled-D labels have no compile bridge.

In [ ]:
source = build_integration_source()
fitted = reusable_rules(provider, source, sample, n_bins=headline_bins, soft_steps=soft_steps)
rules = {row.key: row for row in fitted.rows}
for row in rules.values():
    print(
        f"{row.label:32s} profiled_retention={row.profiled_retention:.4f}  "
        f"own_criterion={row.criterion_efficiency:.4f}  hardening_gap={row.hardening_gap:+.2e}  "
        f"runs={row.n_runs}"
    )

assert rules["d_rule"].hardening_gap == 0.0
# A nearest-centre rule is a restricted family: the finite profiled optimum is not in it.
assert rules["ds_rule"].profiled_retention < profiled_retention(
    sample, np.asarray(headline_profiled_partition.labels), headline_bins
)

## The comb, and the fragments

The score depends on $u$ only through the fringe phase, so a connected score-space cell pulls back to several detector intervals, one wherever the trajectory crosses the cell: the compiled six-cell D rule is a comb of detector runs, not six segments. `run_diagnostics` counts the runs with the two detector ends joined, since $s(0)$ and $s(u_{\max})$ are the same score vector.

The profiled partition combs the detector too, and adds very narrow runs where the efficient score crosses zero steeply. `profiled_trace` shows where they come from -- the certified interval initializer already has them, and the exchange that follows changes few labels and gains little -- and `smoothing_ladder` shows what they are worth, by absorbing every run narrower than a threshold into its wider neighbour and re-measuring the profiled retention.

In [ ]:
d_labels = np.asarray(headline_d_partition.labels)
ds_labels = np.asarray(headline_profiled_partition.labels)
for name, labels in (("D  ", d_labels), ("D_s", ds_labels)):
    geometry = run_diagnostics(sample, labels, n_bins=headline_bins)
    print(
        f"{name}: {geometry.n_runs} runs, per cell {geometry.runs_per_bin}, "
        f"narrowest {geometry.min_run_width:.3f}"
    )
d_geometry = run_diagnostics(sample, d_labels, n_bins=headline_bins)
assert d_geometry.n_runs > headline_bins
assert sum(d_geometry.runs_per_bin) == d_geometry.n_runs

reference = unbinned_profiled_information(sample.scores, sample.weights)
headline = sweep_bin_budget(sample, reference, budgets=(headline_bins,))[0]
trace = profiled_trace(sample, headline)
print(
    f"initializer: {trace.initial.n_runs} runs, retention {trace.initial_retention:.4f}; "
    f"exchange: {trace.accepted_moves} moves in {trace.scans} scans, "
    f"relabelled {trace.relabelled_fraction:.2%}, retention gain {trace.retention_gain:.1e}"
)
assert trace.exchange_stable
assert 0.0 <= trace.retention_gain < 1e-2

print(f"{'min width':>9}  {'runs':>4}  {'bins':>4}  {'retention':>9}")
for row in smoothing_ladder(sample, headline.profiled_labels, n_bins=headline_bins):
    print(f"{row.min_width:>9.2f}  {row.n_runs:>4}  {row.bins_used:>4}  {row.retention:>9.4f}")
    assert row.retention <= trace.final_retention + 1e-12

## Figures

The full study (`run_study`, which the cells above reproduce piece by piece) draws both acts: the D-optimal partition with the cells of its compiled Mahalanobis-Voronoi rule shaded in score space and the comb along the detector, then the profiled partition with the efficient score along the detector and the three detector strips -- the efficient-score initializer, the exchange-stable profiled partition and the reusable soft-Voronoi rule.

In [ ]:
study = run_study(n_nodes=n_nodes, soft_steps=soft_steps, budgets=budgets)
make_d_geometry_figure(study)
make_profiled_figure(study)
plt.show()